In [1]:
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [24]:
anuncios = ['tv','revista']
requisitos = ['sedas','utilitarios','caminhoes']
valores_minimos = {
    requisitos[0]:1.03,
    requisitos[1]:1.14,
    requisitos[2]:1.04,
}

informacoes ={
    anuncios[0]:{
        requisitos[0]:1,
        requisitos[1]:1.03,
        requisitos[2]:0.99,
    },
    anuncios[1]:{
        requisitos[0]:1.02,
        requisitos[1]:1.01,
        requisitos[2]:1.04,
    }
}

custo={
    anuncios[0]:500000,
    anuncios[1]:750000,
    }


In [25]:
model=pyo.ConcreteModel()

model.anuncios = pyo.Set(initialize=anuncios)
model.requisitos = pyo.Set(initialize=requisitos)
model.x = pyo.Var(model.anuncios, domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(model.x[a] * custo[a] for a in model.anuncios)

model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

def restricoes(model, r):
    return sum(model.x[a] * informacoes[a][r] for a in model.anuncios) >= valores_minimos[r]
model.restricoes = pyo.Constraint(model.requisitos, rule=restricoes)



In [26]:
model.pprint()

2 Set Declarations
    anuncios : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'tv', 'revista'}
    requisitos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'sedas', 'utilitarios', 'caminhoes'}

1 Var Declarations
    x : Size=2, Index=anuncios
        Key     : Lower : Value : Upper : Fixed : Stale : Domain
        revista :     0 :  None :  None : False :  True : NonNegativeIntegers
             tv :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 500000*x[tv] + 750000*x[revista]

1 Constraint Declarations
    restricoes : Size=3, Index=requisitos, Active=True
        Key         : Lower : Body                         : Upper : Active
          caminhoes :  1.04 : 0.9

In [27]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpt3qklg1f.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp6xdsxp43.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp6xdsxp43.pyomo.lp
Objective sense      : Minimize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Greater: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 500000.0    

In [28]:
for m in model.x:
    print(f'{m} = {model.x[m].value}')

print(f'Custo total: {model.obj()}')

tv = 2.0
revista = 0.0
Custo total: 1000000.0


In [29]:
model.restricoes.display()

restricoes : Size=3
    Key         : Lower : Body : Upper
      caminhoes :  1.04 : 1.98 :  None
          sedas :  1.03 :  2.0 :  None
    utilitarios :  1.14 : 2.06 :  None
